# nginx — Reverse Proxy & Stream Config

nginx can act as an HTTP reverse proxy (L7) or a raw TCP/UDP stream proxy (L4) via the stream module.

## Generate a reverse-proxy nginx.conf

In [ ]:
template = """
user  nginx;
worker_processes  auto;
error_log  /var/log/nginx/error.log notice;

events {{ worker_connections 1024; }}

http {{
    upstream {upstream_name} {{
        {lb_algorithm};
        {servers}
        keepalive 32;
    }}

    server {{
        listen {listen_port};
        server_name {server_name};

        location / {{
            proxy_pass         http://{upstream_name};
            proxy_http_version 1.1;
            proxy_set_header   Host              $host;
            proxy_set_header   X-Real-IP         $remote_addr;
            proxy_set_header   X-Forwarded-For   $proxy_add_x_forwarded_for;
            proxy_set_header   Connection        "";
        }}
    }}
}}
""".strip()

params = dict(
    upstream_name="app_backend",
    lb_algorithm="least_conn",
    servers="\n        ".join(f"server app{i}.internal:8080;" for i in range(1, 4)),
    listen_port=80,
    server_name="app.example.com",
)
conf = template.format(**params)
print(conf)

import pathlib
pathlib.Path("/tmp/nginx-generated.conf").write_text(conf)

## Validate with nginx -t

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "run", "--rm",
     "-v", "/tmp/nginx-generated.conf:/etc/nginx/nginx.conf:ro",
     "nginx:alpine", "nginx", "-t"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

## Stream (TCP) Config Generator

In [ ]:
def stream_config(tcp_forwards):
    lines = ["stream {"]
    for local_port, remote_host, remote_port in tcp_forwards:
        lines += [
            f"    upstream up_{local_port} {{ server {remote_host}:{remote_port}; }}",
            f"    server {{ listen {local_port}; proxy_pass up_{local_port}; }}",
            "",
        ]
    lines.append("}")
    return "\n".join(lines)

print(stream_config([
    (3306, "db.internal", 3306),
    (5432, "pg.internal", 5432),
    (6379, "redis.internal", 6379),
]))